<a href="https://colab.research.google.com/github/nihal-kabir/CSE425-Neural-Networks/blob/main/425_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# Cell 1
!pip install beir sentence-transformers rank_bm25 faiss-cpu

In [ ]:
# Cell 2
import math, time, os, warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, field
from typing import Dict, List
import numpy as np

DATASETS      = ["nfcorpus", "scifact", "trec-covid", "fiqa"]
SPLIT         = "test"
TOP_K         = 10
RERANK_POOL   = 50
BI_ENCODER    = "all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
USE_FAISS     = False
BATCH_SIZE    = 64

print(f"Config → datasets={DATASETS}  split={SPLIT}  top_k={TOP_K}")
print(f"         bi-encoder={BI_ENCODER}")

In [ ]:
# Cell 3
import os, zipfile, requests
from beir.datasets.data_loader import GenericDataLoader

DATA_DIR = "./datasets"
os.makedirs(DATA_DIR, exist_ok=True)

def load_beir_dataset(dataset_name: str, split: str = SPLIT):
    """Download (if needed) and load a BEIR dataset; returns corpus, queries, qrels."""
    data_path = os.path.join(DATA_DIR, dataset_name)
    if not os.path.isdir(data_path):
        url      = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"
        zip_path = os.path.join(DATA_DIR, f"{dataset_name}.zip")
        print(f"Downloading {dataset_name} …")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(DATA_DIR)
    else:
        print(f"Using cached {dataset_name}")
    corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split=split)
    print(f"  → {len(corpus):,} docs | {len(queries):,} queries | "
          f"{sum(len(v) for v in qrels.values()):,} qrels")
    return corpus, queries, qrels

print("load_beir_dataset() ready.")

In [ ]:
# Cell 4
import random
random.seed(42)

corpus, queries, qrels = load_beir_dataset("nfcorpus")

sample_doc_id = random.choice(list(corpus.keys()))
doc = corpus[sample_doc_id]
print("── Sample document ──────────────────────────────────────────")
print(f"ID    : {sample_doc_id}")
print(f"Title : {doc['title']}")
print(f"Text  : {doc['text'][:300]} …")

sample_qid = random.choice(list(qrels.keys()))
print("\n── Sample query & qrels ─────────────────────────────────────")
print(f"Query : {queries[sample_qid]}")
print(f"Relevant doc IDs: {list(qrels[sample_qid].keys())[:5]}")

doc_lens = [len(d["text"].split()) for d in corpus.values()]
print("\n── Corpus stats ─────────────────────────────────────────────")
print(f"  Avg doc length : {np.mean(doc_lens):.0f} tokens")
print(f"  Max doc length : {np.max(doc_lens)} tokens")
print(f"  Avg relevant/q : {np.mean([len(v) for v in qrels.values()]):.1f}")

In [ ]:
# Cell 5
@dataclass
class RetrievalResult:
    query_id: str
    ranked_doc_ids: List[str]
    scores: List[float]

def get_relevant(qid: str) -> List[str]:
    """Returns relevant doc IDs (binary, score>0) for MRR/Recall/P/MAP.
    References the current global `qrels`."""
    return [doc_id for doc_id, score in qrels.get(qid, {}).items() if score > 0]

def get_grades(qid: str) -> Dict[str, int]:
    """Returns {doc_id: relevance_grade} for graded NDCG (e.g. trec-covid 0/1/2).
    Keeps only positive grades; references the current global `qrels`."""
    return {doc_id: score for doc_id, score in qrels.get(qid, {}).items() if score > 0}

print("RetrievalResult, get_relevant() and get_grades() defined.")

In [ ]:
# Cell 6
def reciprocal_rank(ranked: List[str], relevant: List[str]) -> float:
    for rank, doc_id in enumerate(ranked, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def recall_at_k(ranked: List[str], relevant: List[str], k: int) -> float:
    return len(set(ranked[:k]) & set(relevant)) / max(len(relevant), 1)

def precision_at_k(ranked: List[str], relevant: List[str], k: int) -> float:
    return len(set(ranked[:k]) & set(relevant)) / k if k else 0.0

def average_precision(ranked: List[str], relevant: List[str]) -> float:
    hits, total = 0, 0.0
    for rank, doc_id in enumerate(ranked, 1):
        if doc_id in relevant:
            hits += 1
            total += hits / rank
    return total / max(len(relevant), 1)

def ndcg_at_k(ranked: List[str], grades: Dict[str, int], k: int) -> float:
    """Graded NDCG@k with linear gains (gain = relevance grade), matching
    trec_eval / pytrec_eval's ndcg_cut used by the official BEIR benchmark."""
    dcg   = sum(grades.get(d, 0) / math.log2(r + 1)
                for r, d in enumerate(ranked[:k], 1))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg  = sum(g / math.log2(r + 1) for r, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

def evaluate(results: List[RetrievalResult], k: int = TOP_K) -> Dict[str, float]:
    mrr, rec, prec, ap, ndcg = [], [], [], [], []
    for res in results:
        rel    = get_relevant(res.query_id)
        grades = get_grades(res.query_id)
        if not rel:
            continue
        mrr.append(reciprocal_rank(res.ranked_doc_ids[:k], rel))
        rec.append(recall_at_k(res.ranked_doc_ids, rel, k))
        prec.append(precision_at_k(res.ranked_doc_ids, rel, k))
        ap.append(average_precision(res.ranked_doc_ids[:k], rel))
        ndcg.append(ndcg_at_k(res.ranked_doc_ids, grades, k))
    n = len(mrr)
    if n == 0:
        return {m: 0.0 for m in [f"MRR@{k}", f"Recall@{k}", f"P@{k}", f"MAP@{k}", f"NDCG@{k}"]} | {"N queries": 0}
    return {
        f"MRR@{k}":    round(sum(mrr)/n,  4),
        f"Recall@{k}": round(sum(rec)/n,  4),
        f"P@{k}":      round(sum(prec)/n, 4),
        f"MAP@{k}":    round(sum(ap)/n,   4),
        f"NDCG@{k}":   round(sum(ndcg)/n, 4),
        "N queries":   n,
    }

def per_query_ndcg(results: List[RetrievalResult], k: int = TOP_K) -> Dict[str, float]:
    """Returns {query_id: graded ndcg@k} for every query with at least one relevant doc."""
    scores = {}
    for res in results:
        grades = get_grades(res.query_id)
        if grades:
            scores[res.query_id] = ndcg_at_k(res.ranked_doc_ids, grades, k)
    return scores

def print_metrics(label: str, m: Dict):
    print(f"\n{'='*52}")
    print(f"  {label}")
    print(f"{'='*52}")
    for k, v in m.items():
        if isinstance(v, float):
            bar = "█" * int(v * 30)
            print(f"  {k:<14}  {v:.4f}  {bar}")
        else:
            print(f"  {k:<14}  {v}")

print("Evaluation functions ready (graded NDCG).")

In [ ]:
# Cell 7
from rank_bm25 import BM25Okapi

class BM25Retriever:
    def __init__(self, doc_ids: List[str], doc_texts: List[str], k1=1.5, b=0.75):
        self.doc_ids = doc_ids
        tokenized    = [t.lower().split() for t in doc_texts]
        self.bm25    = BM25Okapi(tokenized, k1=k1, b=b)
        print(f"[BM25] Indexed {len(doc_ids):,} documents")

    def retrieve(self, query_id: str, query_text: str, k: int = TOP_K) -> RetrievalResult:
        scores  = self.bm25.get_scores(query_text.lower().split())
        top_idx = np.argsort(scores)[::-1][:k]
        return RetrievalResult(query_id, [self.doc_ids[i] for i in top_idx], scores[top_idx].tolist())

    def retrieve_all(self, qids, qtexts, k=TOP_K):
        return [self.retrieve(qid, qt, k) for qid, qt in zip(qids, qtexts)]

In [ ]:
# Cell 8
from sentence_transformers import SentenceTransformer

class DenseRetriever:
    def __init__(self, doc_ids: List[str], doc_texts: List[str],
                 model_name=BI_ENCODER, use_faiss=USE_FAISS):
        self.doc_ids   = doc_ids
        self.use_faiss = use_faiss

        print(f"[Dense] Loading model: {model_name}")
        self.model = SentenceTransformer(model_name)

        print(f"[Dense] Encoding {len(doc_ids):,} documents …")
        t0 = time.time()
        self.doc_embs = self.model.encode(
            doc_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype("float32")
        print(f"[Dense] Done in {time.time()-t0:.1f}s  |  dim={self.doc_embs.shape[1]}")

        if use_faiss:
            self._build_faiss()

    def _build_faiss(self):
        try:
            import faiss
        except ImportError:
            raise ImportError("Run: pip install faiss-cpu")
        dim        = self.doc_embs.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.doc_embs)
        print(f"[FAISS] Index built — {self.index.ntotal:,} vectors")

    def retrieve(self, query_id: str, query_text: str, k: int = TOP_K) -> RetrievalResult:
        q = self.model.encode([query_text], normalize_embeddings=True,
                               convert_to_numpy=True).astype("float32")
        if self.use_faiss:
            scores, idxs = self.index.search(q, k)
            scores, idxs = scores[0], idxs[0]
        else:
            scores = (self.doc_embs @ q.T).squeeze()
            idxs   = np.argsort(scores)[::-1][:k]
            scores = scores[idxs]
        return RetrievalResult(query_id, [self.doc_ids[i] for i in idxs], scores.tolist())

    def retrieve_all(self, qids, qtexts, k=TOP_K):
        q_embs = self.model.encode(
            list(qtexts),
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype("float32")
        if self.use_faiss:
            scores_mat, idxs_mat = self.index.search(q_embs, k)
            return [
                RetrievalResult(qid, [self.doc_ids[i] for i in idxs], scores.tolist())
                for qid, scores, idxs in zip(qids, scores_mat, idxs_mat)
            ]
        sim = self.doc_embs @ q_embs.T
        results = []
        for j, qid in enumerate(qids):
            scores = sim[:, j]
            idxs   = np.argsort(scores)[::-1][:k]
            results.append(RetrievalResult(qid, [self.doc_ids[i] for i in idxs], scores[idxs].tolist()))
        return results

In [ ]:
# Cell 9
from sentence_transformers.cross_encoder import CrossEncoder

class CrossEncoderReranker:
    def __init__(self, model_name=CROSS_ENCODER):
        print(f"[Reranker] Loading: {model_name}")
        self.model = CrossEncoder(model_name)

    def rerank(self, query_id: str, query_text: str,
               first_stage: RetrievalResult, k: int = TOP_K) -> RetrievalResult:
        candidates = first_stage.ranked_doc_ids
        texts      = [corpus[d]["title"] + " " + corpus[d]["text"] for d in candidates]
        pairs      = [(query_text, t) for t in texts]
        scores     = self.model.predict(pairs)
        ranked_idx = np.argsort(scores)[::-1][:k]
        return RetrievalResult(
            query_id,
            [candidates[i] for i in ranked_idx],
            [float(scores[i]) for i in ranked_idx],
        )

    def rerank_all(self, qids, qtexts, first_stage_results, k=TOP_K):
        from tqdm import tqdm
        return [
            self.rerank(qid, qt, fs, k)
            for qid, qt, fs in tqdm(zip(qids, qtexts, first_stage_results), total=len(qids))
        ]

reranker = CrossEncoderReranker()

In [ ]:
# Cell 10
import pandas as pd

all_dataset_results = {}
all_dataset_ndcg    = {}

for dataset_name in DATASETS:
    print(f"\n{'#'*62}")
    print(f"#  {dataset_name.upper()}")
    print(f"{'#'*62}")

    corpus, queries, qrels = load_beir_dataset(dataset_name)

    corpus_ids   = list(corpus.keys())
    corpus_texts = [(corpus[d]["title"] + " " + corpus[d]["text"]).strip()
                    for d in corpus_ids]
    query_ids    = list(queries.keys())
    query_texts  = [queries[qid] for qid in query_ids]

    bm25_r   = BM25Retriever(corpus_ids, corpus_texts)
    bm25_res = bm25_r.retrieve_all(query_ids, query_texts, k=TOP_K)
    bm25_m   = evaluate(bm25_res)

    dense_r   = DenseRetriever(corpus_ids, corpus_texts)
    dense_res = dense_r.retrieve_all(query_ids, query_texts, k=TOP_K)
    dense_m   = evaluate(dense_res)

    expanded     = dense_r.retrieve_all(query_ids, query_texts, k=RERANK_POOL)
    reranked_res = reranker.rerank_all(query_ids, query_texts, expanded, k=TOP_K)
    reranked_m   = evaluate(reranked_res)

    hth = pd.DataFrame({
        "BM25":         bm25_m,
        "Dense":        dense_m,
        "Dense+Rerank": reranked_m,
    }).T
    metric_cols = [c for c in hth.columns if c != "N queries"]
    try:
        from IPython.display import display as _display
        _display(hth[metric_cols].astype(float).round(4).style
            .background_gradient(cmap="YlGn", axis=0)
            .format("{:.4f}")
            .set_caption(f"Metrics @{TOP_K}  |  {dataset_name}  |  {bm25_m['N queries']} queries"))
    except Exception:
        print(hth[metric_cols].to_string())

    all_dataset_results[dataset_name] = {
        "BM25":         bm25_m,
        "Dense":        dense_m,
        "Dense+Rerank": reranked_m,
    }
    all_dataset_ndcg[dataset_name] = {
        "BM25":         per_query_ndcg(bm25_res),
        "Dense":        per_query_ndcg(dense_res),
        "Dense+Rerank": per_query_ndcg(reranked_res),
    }

    bm25_results, dense_results, reranked_results = bm25_res, dense_res, reranked_res
    bm25_retriever, dense_retriever               = bm25_r, dense_r

print("\n✓ All datasets complete.")

In [ ]:
# Cell 11
metric_cols = [f"MRR@{TOP_K}", f"Recall@{TOP_K}", f"P@{TOP_K}", f"MAP@{TOP_K}", f"NDCG@{TOP_K}"]

rows = []
for dataset_name, methods in all_dataset_results.items():
    for method, m in methods.items():
        row = {"Dataset": dataset_name, "Method": method}
        row.update({col: m[col] for col in metric_cols})
        rows.append(row)

summary_df = (
    pd.DataFrame(rows)
    .set_index(["Dataset", "Method"])
    [metric_cols]
    .astype(float)
    .round(4)
)

print("\n" + "="*70)
print("  Cross-Dataset Summary — All Methods")
print("="*70)
try:
    from IPython.display import display as _display
    _display(summary_df.style
        .background_gradient(cmap="YlGn", axis=0)
        .format("{:.4f}")
        .set_caption(f"Metrics @{TOP_K} — all datasets (rows = dataset × method)"))
except Exception:
    print(summary_df.to_string())

In [ ]:
# Cell 12
from scipy.stats import wilcoxon

print(f"Paired Wilcoxon Signed-Rank Tests — NDCG@{TOP_K}\n")
print(f"{'Dataset':<15}  {'Pair':<25}  {'W':>10}  {'p-value':>10}  Verdict")
print("-" * 82)

for dataset_name, ndcg_by_method in all_dataset_ndcg.items():
    bm25_d   = ndcg_by_method["BM25"]
    dense_d  = ndcg_by_method["Dense"]
    rerank_d = ndcg_by_method["Dense+Rerank"]

    common   = sorted(set(bm25_d) & set(dense_d) & set(rerank_d))
    bm25_v   = np.array([bm25_d[q]   for q in common])
    dense_v  = np.array([dense_d[q]  for q in common])
    rerank_v = np.array([rerank_d[q] for q in common])

    for pair_label, a, b in [
        ("BM25 ↔ Dense",         bm25_v,  dense_v),
        ("Dense ↔ Dense+Rerank", dense_v, rerank_v),
    ]:
        diff = b - a
        if np.any(diff != 0):
            stat, p = wilcoxon(diff, alternative="two-sided")
        else:
            stat, p = float("nan"), 1.0
        verdict = "significant (p<0.05)" if p < 0.05 else "not significant"
        print(f"{dataset_name:<15}  {pair_label:<25}  {stat:>10.1f}  {p:>10.4f}  {verdict}")
    print()

In [ ]:
# Cell 13
random.seed(0)
sample_qid = random.choice(query_ids)
sample_qt  = queries[sample_qid]
rel_ids    = set(get_relevant(sample_qid))

print(f"Query  : {sample_qt}")
print(f"Relevant docs ({len(rel_ids)} total): {list(rel_ids)[:4]}\n")

for label, results in [
    ("BM25",         bm25_results),
    ("Dense",        dense_results),
    ("Dense+Rerank", reranked_results),
]:
    res  = next(r for r in results if r.query_id == sample_qid)
    hits = ["✓" if d in rel_ids else "✗" for d in res.ranked_doc_ids[:5]]
    print(f"[{label}] Top-5:")
    for rank, (doc_id, score, hit) in enumerate(zip(res.ranked_doc_ids[:5], res.scores[:5], hits), 1):
        title = corpus[doc_id]["title"][:60]
        print(f'  {rank}. [{hit}] ({score:.3f})  "{title}"')
    print()

In [ ]:
# Cell 14
import matplotlib.pyplot as plt

CURVE_K = 100
ks      = [1, 3, 5, 10, 20, 50, 100]

print("[Curve] Running k=100 retrieval for NDCG curve …")
bm25_curve   = bm25_retriever.retrieve_all(query_ids, query_texts, k=CURVE_K)
dense_curve  = dense_retriever.retrieve_all(query_ids, query_texts, k=CURVE_K)
_pool_curve  = dense_retriever.retrieve_all(query_ids, query_texts, k=max(RERANK_POOL, CURVE_K))
rerank_curve = reranker.rerank_all(query_ids, query_texts, _pool_curve, k=CURVE_K)
print("[Curve] Done.")

def ndcg_curve(results, k_list):
    return [evaluate(results, k=k)[f"NDCG@{k}"] for k in k_list]

plt.figure(figsize=(8, 4))
for label, results in [
    ("BM25",            bm25_curve),
    ("Dense Bi-Encoder",dense_curve),
    ("Dense+Rerank",    rerank_curve),
]:
    plt.plot(ks, ndcg_curve(results, ks), marker="o", label=label)

plt.xlabel("K"); plt.ylabel("NDCG@K")
plt.title(f"NDCG@K Across Retrieval Methods — {dataset_name}")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("ndcg_curve.png", dpi=150)
plt.show()
print("Saved: ndcg_curve.png")